# 🛰️ Satellite Imagery Downloader — India

Downloads satellite imagery for any Indian state using Google Earth Engine,
and produces a single analysis-ready clipped GeoTIFF.

> **Before running:** Complete the setup steps in [`SETUP.md`](SETUP.md)  
> **Then:** Update the 3 variables in Cell 0 below and run all cells top to bottom.

In [118]:
import os

# ============================================================
#  ⚙️ USER CONFIGURATION — update these before running
# ============================================================

# Path to the folder where downloaded imagery will be saved
BASE_OUTPUT_DIR = r"C:\Users\Subham V Sharma\Desktop\Python geospatial\satellite-downloader\satellite-imagery-downloader"

# Path to the india_state.shp file (included in this repo)
# Just replace this with the path to wherever you cloned the repo
SHAPEFILE_PATH = r"C:\Users\Subham V Sharma\Desktop\Python geospatial\satellite-downloader\satellite-imagery-downloader\india_state.shp"

# Your Google Earth Engine project ID (you need to set up a GEE account and create a project to get this)
GEE_PROJECT_ID = 'sentinel2-project-498918' 
# example: 'sentinel2-project-498918' (put your own project ID here)

# ============================================================

In [119]:
import ee

try:
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

print("Earth Engine Initialized")
import geemap
print("Earth Engine Initialized")


state_name = input("Enter state name: ").strip()
print(f"Selected state: {state_name}")

Earth Engine Initialized
Earth Engine Initialized
Selected state: sikkim


In [120]:
# ==========================================
# SATELLITE MANAGER
# ==========================================

satellites = {

    "Sentinel-2": {
        "dataset": "COPERNICUS/S2_HARMONIZED",
        "cloud_field": "CLOUDY_PIXEL_PERCENTAGE",
        "data_type_bytes": 2,
        "bands": [
            "B1","B2","B3","B4",
            "B5","B6","B7","B8",
            "B8A","B9","B11","B12"
        ],
        "resolution": {
            "10m": ["B2","B3","B4","B8"],
            "20m": ["B5","B6","B7","B8A","B11","B12"],
            "60m": ["B1","B9"]
        }
    },

    "Landsat-8": {
        "dataset": "LANDSAT/LC08/C02/T1_L2",
        "cloud_field": "CLOUD_COVER",
        "data_type_bytes": 2,
        "bands": [
            "SR_B1","SR_B2","SR_B3",
            "SR_B4","SR_B5","SR_B6",
            "SR_B7"
        ],
        "resolution": "30m"
    },

    "Landsat-9": {
        "dataset": "LANDSAT/LC09/C02/T1_L2",
        "cloud_field": "CLOUD_COVER",
        "data_type_bytes": 2,
        "bands": [
            "SR_B1","SR_B2","SR_B3",
            "SR_B4","SR_B5","SR_B6",
            "SR_B7"
        ],
        "resolution": "30m"
    }
}

---
## 🛰️ Satellite & Band Selection

Select your satellite and the bands you want to download.
You will be prompted to enter choices by number.

**Sentinel-2 quick reference:**
| Band | Description | Resolution |
|------|-------------|------------|
| B2 | Blue | 10m |
| B3 | Green | 10m |
| B4 | Red | 10m |
| B8 | NIR | 10m |
| B5, B6, B7, B8A | Red Edge | 20m |
| B11, B12 | SWIR | 20m |
| B1 | Coastal Aerosol | 60m |
| B9 | Water Vapour | 60m |

> **Tip:** For natural colour RGB enter `4, 3, 2`. For false colour with NIR enter `4, 3, 2, 8`.

In [121]:
print("Available Satellites:\n")

for i, sat in enumerate(satellites, start=1):
    print(f"{i}. {sat}")

choice = int(
    input("\nEnter satellite number (1=Sentinel-2, 2=Landsat-8, 3=Landsat-9) : ")
)

Available Satellites:

1. Sentinel-2
2. Landsat-8
3. Landsat-9


In [122]:
satellite_name = list(satellites.keys())[choice - 1]
print("Selected Satellite:")
print(satellite_name)

Selected Satellite:
Landsat-8


In [123]:
satellite_info = satellites[satellite_name]

dataset = satellite_info["dataset"]

available_bands = satellite_info["bands"]
print("Dataset:")
print(dataset)

resolution_info = satellite_info["resolution"]

print("Resolution Info:")

if isinstance(resolution_info, dict):

    for reso, bands in resolution_info.items():

        print(f"{reso}: {bands}")

else:

    print(resolution_info)

print("Available Bands:\n")

for i, band in enumerate(available_bands, start=1):
    print(f"{i}. {band}")


Dataset:
LANDSAT/LC08/C02/T1_L2
Resolution Info:
30m
Available Bands:

1. SR_B1
2. SR_B2
3. SR_B3
4. SR_B4
5. SR_B5
6. SR_B6
7. SR_B7


In [124]:
band_choice = input(
    "\n" +
    "\n".join(
        f"{i}={band},"
        for i, band in enumerate(available_bands, start=1)
    ) +
    "\n\nEnter band numbers separated by commas: "
)

In [125]:
selected_bands = []

band_numbers = band_choice.replace(",", " ").split()

for x in band_numbers:

    band_index = int(x) - 1

    selected_bands.append(
        available_bands[band_index]
    )

In [126]:
print("\nSelected Bands:\n")

for band in selected_bands:
    print(band)

print("\nNumber of Bands:")
print(len(selected_bands))


Selected Bands:

SR_B4
SR_B3
SR_B2

Number of Bands:
3


In [127]:
selected_resolutions = []

if isinstance(satellite_info["resolution"], dict):

    for res, bands in satellite_info["resolution"].items():

        for band in selected_bands:

            if band in bands:

                selected_resolutions.append(
                    int(res.replace("m", ""))
                )

    resolution = min(selected_resolutions)

else:

    resolution = int(
        satellite_info["resolution"].replace("m", "")
    )

print(
    "Working Resolution:",
    resolution,
    "meters"
)

Working Resolution: 30 meters


In [128]:
safe_state = state_name.replace(" ", "_")

folder_name = f"{safe_state}_{satellite_name}"

print(folder_name)
download_path = os.path.join(BASE_OUTPUT_DIR, folder_name)
os.makedirs(
    download_path,
    exist_ok=True
)

sikkim_Landsat-8


---
## 📅 Date Range & Cloud Filter

Enter the start and end dates for the imagery search, and a maximum cloud cover percentage.

- **Tip:** Use a 30–45 day window with cloud threshold 10–20% for best results.
- **Put** date in YYYY MM DD.
- **A** very narrow date range may return too few images to cover the full state.

In [129]:
start_date = input(
    "Enter start date(YYYY MM DD): "
).strip()

start_date = start_date.replace("/", "-")
start_date = start_date.replace(" ", "-")
start_date = start_date.replace(",", "-")

In [130]:
end_date = input(
    "Enter end date(YYYY MM DD): "
).strip()

end_date = end_date.replace("/", "-")
end_date = end_date.replace(",", "-")
end_date = end_date.replace(" ", "-")

In [131]:
from datetime import datetime

try:
    datetime.strptime(start_date, "%Y-%m-%d")
    datetime.strptime(end_date, "%Y-%m-%d")

except ValueError:

    raise ValueError(
        "Date must be in YYYY-MM-DD format"
    )

In [132]:
cloud_threshold = float(
    input(
        "\nEnter maximum cloud percentage (0-100): "
    )
)
if cloud_threshold < 0 or cloud_threshold > 100:

    raise ValueError(
        "Cloud threshold must be between 0 and 100."
    )

In [133]:
print("\n========================")
print("  DOWNLOAD SETTINGS")
print("========================")

print(f"State: {state_name}")

print(f"Satellite: {satellite_name}")

print(f"Dataset: {dataset}")

print(f"Start Date: {start_date}")

print(f"End Date: {end_date}")

print(f"Cloud Threshold: {cloud_threshold}%")

print(f"Bands: {selected_bands}")

print(f"Number of Bands: {len(selected_bands)}")


  DOWNLOAD SETTINGS
State: sikkim
Satellite: Landsat-8
Dataset: LANDSAT/LC08/C02/T1_L2
Start Date: 2026-01-01
End Date: 2026-03-25
Cloud Threshold: 10.0%
Bands: ['SR_B4', 'SR_B3', 'SR_B2']
Number of Bands: 3


---
## 🗺️ AOI Manager

Loads the state boundary from the shapefile and converts it to a GEE FeatureCollection.
The image collection is then filtered to only include scenes that intersect the state,
and a median composite is created from all valid images in the date range.

> **Note:** If `Images Found` prints `0`, widen your date range or increase the cloud threshold.

In [134]:
import geopandas as gpd

states = gpd.read_file(SHAPEFILE_PATH)


state_gdf = states[
    states["State_Name"].str.strip().str.lower()
    ==
    state_name.strip().lower()
]

state = geemap.geopandas_to_ee(state_gdf)

In [135]:
state_geometry = state.geometry()

In [136]:
bounding_box = state_geometry.bounds()

In [137]:
collection = (
    ee.ImageCollection(dataset)
    .filterBounds(state.geometry())
    .filterDate(start_date, end_date)
    .filter(
        ee.Filter.lt(
            satellite_info["cloud_field"],
            cloud_threshold
        )
    )

)

print(
    "Images Found:",
    collection.size().getInfo()
)

Images Found: 9


---
## 🔲 Bounding Box & Tile Grid

Computes the bounding box of the state and generates a grid of tiles over it.
Tiles that don't overlap the actual state boundary are filtered out to avoid unnecessary downloads.

In [138]:
coords = bounding_box.coordinates().getInfo()[0]

min_lon = coords[0][0]
min_lat = coords[0][1]

max_lon = coords[2][0]
max_lat = coords[2][1]

In [139]:
print("Min Longitude:", min_lon)
print("Max Longitude:", max_lon)

print("Min Latitude:", min_lat)
print("Max Latitude:", max_lat)

Min Longitude: 88.01280217041752
Max Longitude: 88.91926028912548
Min Latitude: 27.07936365361849
Max Latitude: 28.12939545121195


In [140]:
width_deg = max_lon - min_lon
height_deg = max_lat - min_lat

print("Width:", width_deg)
print("Height:", height_deg)

Width: 0.9064581187079597
Height: 1.0500317975934585


---
## 🧩 Tile Size & Chunk Calculator

Google Earth Engine has a ~48MB limit per direct download request.
This section calculates the maximum safe tile size based on band count, bit depth, and resolution,
then splits bands into chunks of 3 to stay within the pixel size limit per request.

In [141]:
band_chunks = []

for i in range(
    0,
    len(selected_bands),
    3
):

    chunk = selected_bands[i:i+3]

    band_chunks.append(chunk)

print("\nBand Chunks:\n")

for i, chunk in enumerate(
    band_chunks,
    start=1
):
    print(
        f"Chunk {i}: {chunk}"
    )

largest_chunk_size = max(
    len(chunk)
    for chunk in band_chunks
)

data_type_bytes = satellite_info[
    "data_type_bytes"
]

print(
    "\nLargest Chunk Size:",
    largest_chunk_size
)

print(band_chunks)


Band Chunks:

Chunk 1: ['SR_B4', 'SR_B3', 'SR_B2']

Largest Chunk Size: 3
[['SR_B4', 'SR_B3', 'SR_B2']]


In [142]:
import math

# Earth Engine direct download limit
max_download_bytes = 50_331_648

bytes_per_pixel = (
    largest_chunk_size
    *
    data_type_bytes
)

max_pixels = (
    max_download_bytes
    /
    bytes_per_pixel
)

max_dimension_pixels = math.sqrt(
    max_pixels
)

tile_size_meters = (
    max_dimension_pixels
    *
    resolution
)

tile_size_km = (
    tile_size_meters
    / 1000
)

safety_factor = 0.58

tile_size_km = (
    tile_size_km
    *
    safety_factor
)

print("\nTile Size Calculator\n")

print(
    "Largest Chunk Size:",
    largest_chunk_size
)

print(
    "Bytes Per Pixel:",
    bytes_per_pixel
)

print(
    "Maximum Pixels:",
    int(max_pixels)
)

print(
    "Maximum Dimension (Pixels):",
    int(max_dimension_pixels)
)

print(
    f"Safe Tile Size: {tile_size_km:.2f} km"
)


Tile Size Calculator

Largest Chunk Size: 3
Bytes Per Pixel: 6
Maximum Pixels: 8388608
Maximum Dimension (Pixels): 2896
Safe Tile Size: 50.40 km


In [143]:
tile_size_deg = (
    tile_size_km
    / 111
)

print(
    "Tile Size (Degrees):",
    tile_size_deg
)

Tile Size (Degrees): 0.45401606430520464


In [144]:
columns = math.ceil(
    width_deg
    /
    tile_size_deg
)

rows = math.ceil(
    height_deg
    /
    tile_size_deg
)

total_tiles = (
    rows
    *
    columns
)

print("\nGrid Information\n")

print(
    "Rows:",
    rows
)

print(
    "Columns:",
    columns
)

print(
    "Total Tiles:",
    total_tiles
)


Grid Information

Rows: 3
Columns: 2
Total Tiles: 6


In [145]:
tile_geometries = []

for row in range(rows):

    for col in range(columns):

        xmin = (
            min_lon
            +
            col * tile_size_deg
        )

        xmax = min(
            xmin + tile_size_deg,
            max_lon
        )

        ymin = (
            min_lat
            +
            row * tile_size_deg
        )

        ymax = min(
            ymin + tile_size_deg,
            max_lat
        )

        tile = ee.Geometry.Rectangle(
            [
                xmin,
                ymin,
                xmax,
                ymax
            ]
        )

        tile_geometries.append(tile)

print(
    "Generated Tiles:",
    len(tile_geometries)
)

Generated Tiles: 6


In [146]:
print(
    "Filtering Tiles..."
)

tiles_fc = ee.FeatureCollection(
    [
        ee.Feature(tile)
        for tile in tile_geometries
    ]
)

filtered_fc = (
    tiles_fc.filterBounds(
        state.geometry()
    )
)

filtered_tiles = [
    ee.Feature(feature).geometry()
    for feature in filtered_fc.getInfo()["features"]
]

print(
    "Tile Filtering Summary"
)

print(
    "Original Tiles:",
    len(tile_geometries)
)

print(
    "Filtered Tiles:",
    len(filtered_tiles)
)

print(
    "Removed Tiles:",
    len(tile_geometries)
    -
    len(filtered_tiles)
)

Filtering Tiles...
Tile Filtering Summary
Original Tiles: 6
Filtered Tiles: 6
Removed Tiles: 0


In [147]:
tiles_fc = ee.FeatureCollection(
    [
        ee.Feature(tile)
        for tile in filtered_tiles
    ]
)

Map = geemap.Map()

Map.centerObject(
    bounding_box,
    7
)

Map.addLayer(
    bounding_box,
    {"color": "red"},
    "Bounding Box"
)

Map.addLayer(
    state,
    {'color': 'blue'},
    'State'
)

Map.addLayer(
    tiles_fc,
    {"color": "green"},
    "Tiles"
)

Map

Map(center=[27.60378639142169, 88.46603122977137], controls=(WidgetControl(options=['position', 'transparent_b…

In [148]:
print("\nGrid Summary\n")

print(
    "Total Tiles Generated:",
    len(tile_geometries)
)

print(
    "Useful Tiles:",
    len(filtered_tiles)
)

print(
    "Tiles Removed:",
    len(tile_geometries) - len(filtered_tiles)
)

print(
    "Band Chunks:",
    len(band_chunks)
)

print(
    "Estimated Downloads:",
    len(filtered_tiles) * len(band_chunks)
)


Grid Summary

Total Tiles Generated: 6
Useful Tiles: 6
Tiles Removed: 0
Band Chunks: 1
Estimated Downloads: 6


In [149]:
tile = filtered_tiles[0]

print(
    "Tile Area km²:",
    tile.area().divide(1e6).getInfo()
)

print(
    tile.coordinates().getInfo()
)

Tile Area km²: 2264.6592479397677
[[[88.01280217041752, 27.07936365361849], [88.46681823472272, 27.07936365361849], [88.46681823472272, 27.533379717923694], [88.01280217041752, 27.533379717923694], [88.01280217041752, 27.07936365361849]]]


---
## ⬇️ Download, Merge & Export

Runs the full download pipeline:
1. Creates a median composite from the image collection
2. Downloads each tile in band chunks
3. Merges band chunks per tile into a single multi-band GeoTIFF
4. Mosaics all tiles into one raster covering the full state
5. Clips the mosaic to the exact state boundary
6. Deletes all intermediate files

**Final output:** `StateName_clipped.tif` in your output folder.

In [150]:
print(
    "\nCreating Base Image..."
)

base_image = (
    collection
    .median()
    .toUint16()
)

export_crs = (
    base_image
    .projection()
    .getInfo()['crs']
)

print(
    "Export CRS:",
    export_crs
)

print(
    "Base Image Ready"
)


Creating Base Image...
Export CRS: EPSG:4326
Base Image Ready


In [151]:

from concurrent.futures import ThreadPoolExecutor
import time
import os

def download_chunk(args):

    i, j, tile, band_chunk = args

    output_file = os.path.join(
        download_path,
        f"tile_{i}_chunk_{j}.tif"
    )

    # Skip if already downloaded
    if os.path.exists(output_file):
        print(f"  ⏭️  Skipping Tile {i} Chunk {j} — already exists")
        return

    for attempt in range(1, 4):  # 3 attempts max

        try:

            print(
                f"  ⬇️  Tile {i} Chunk {j}: {band_chunk} "
                f"(attempt {attempt})..."
            )

            image = (
                base_image
                .select(band_chunk)
                .clip(tile)
            )

            geemap.ee_export_image(
                image,
                filename=output_file,
                crs=export_crs,
                scale=resolution,
                region=tile,
                file_per_band=False
            )

            print(f"  ✅ Done Tile {i} Chunk {j}")
            return  # success — exit retry loop

        except Exception as e:

            print(
                f"  ❌ Failed Tile {i} Chunk {j} "
                f"attempt {attempt}: {e}"
            )

            if attempt < 3:
                time.sleep(5)  # wait 5 sec before retrying

    print(f"  💀 Giving up on Tile {i} Chunk {j} after 3 attempts")


# Build full list of all tile+chunk combinations
tasks = [
    (i, j, tile, band_chunk)
    for i, tile in enumerate(filtered_tiles, start=1)
    for j, band_chunk in enumerate(band_chunks, start=1)
]

print(f"Total downloads: {len(tasks)}")
print(f"Starting parallel download with 4 workers...\n")

# Run 4 downloads at a time
with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(download_chunk, tasks)

print("\n✅ All downloads complete.")

Total downloads: 6
Starting parallel download with 4 workers...

  ⬇️  Tile 1 Chunk 1: ['SR_B4', 'SR_B3', 'SR_B2'] (attempt 1)...
Generating URL ...
  ⬇️  Tile 2 Chunk 1: ['SR_B4', 'SR_B3', 'SR_B2'] (attempt 1)...
Generating URL ...
  ⬇️  Tile 3 Chunk 1: ['SR_B4', 'SR_B3', 'SR_B2'] (attempt 1)...
Generating URL ...
  ⬇️  Tile 4 Chunk 1: ['SR_B4', 'SR_B3', 'SR_B2'] (attempt 1)...
Generating URL ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Data downloaded to C:\Users\Subham V Sharma\Desktop\Python geospatial\satellite-downloader\satellite-imagery-downloader\sikkim_Landsat-8\tile_1_chunk_1.tif
  ✅ Done Tile 1 Chunk 1
  ⬇️  Tile 5 Chunk 1: ['SR_B4', 'SR_B3', 'SR_B2'] (attempt 1)...
Generating URL ...
Data downloaded to C:\Users\Subham V Sharma\Desktop\Python geospatial\satellite-downloader\satellite-imagery-downloader\sikkim_Landsat-8\tile_4_chunk_1.tif
  ✅ Done Tile 4 Chunk 1
  ⬇️  Tile 6 Chunk 1: ['SR_B4', 'SR_B3', 'SR_B2'] (attempt 1)...
Generating URL ...
Data do

In [152]:
from osgeo import gdal
import os
import glob
from concurrent.futures import ThreadPoolExecutor

def merge_tile_chunks(i):

    chunk_files = sorted(
        glob.glob(
            os.path.join(download_path, f"tile_{i}_chunk_*.tif")
        )
    )

    if len(chunk_files) != len(band_chunks):
        print(f"Tile {i}: Missing chunks ({len(chunk_files)}/{len(band_chunks)})")
        return

    output_file = os.path.join(download_path, f"tile_{i}.tif")

    if os.path.exists(output_file):
        print(f"Tile {i}: Already merged, skipping")
        return

    vrt_file = os.path.join(download_path, f"tile_{i}.vrt")

    print(f"Tile {i}: Merging {len(chunk_files)} chunks...")

    gdal.BuildVRT(vrt_file, chunk_files, separate=True)

    gdal.Translate(
        output_file,
        vrt_file,
        format="GTiff",
        creationOptions=[
            "COMPRESS=DEFLATE",
            "PREDICTOR=2",
            "BIGTIFF=YES",
            "TILED=YES"
        ]
    )

    print(f"Tile {i}: ✅ Done")


print(f"Merging {len(filtered_tiles)} tiles in parallel...\n")

with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(merge_tile_chunks, range(1, len(filtered_tiles) + 1))

print("\n✅ All tiles merged.")

Merging 6 tiles in parallel...

Tile 1: Merging 1 chunks...
Tile 2: Merging 1 chunks...
Tile 3: Merging 1 chunks...
Tile 4: Merging 1 chunks...
Tile 3: ✅ Done
Tile 4: ✅ Done
Tile 5: Merging 1 chunks...
Tile 6: Merging 1 chunks...
Tile 1: ✅ Done
Tile 2: ✅ Done
Tile 5: ✅ Done
Tile 6: ✅ Done

✅ All tiles merged.


In [153]:
from osgeo import gdal
import os
import glob
import re

safe_name = state_name.replace(" ", "_")

tile_files = sorted([
    f for f in glob.glob(
        os.path.join(download_path, "tile_*.tif")
    )
    if re.match(r".*tile_\d+\.tif$", f)
])

vrt_file = os.path.join(download_path, f"{safe_name}.vrt")

gdal.BuildVRT(vrt_file, tile_files)

print("VRT created:", vrt_file)

VRT created: C:\Users\Subham V Sharma\Desktop\Python geospatial\satellite-downloader\satellite-imagery-downloader\sikkim_Landsat-8\sikkim.vrt


In [154]:
from osgeo import gdal
import geopandas as gpd
import os

safe_name = state_name.replace(" ", "_")

# Read directly from VRT — no intermediate mosaic tif needed
vrt_file = os.path.join(download_path, f"{safe_name}.vrt")

output_raster = os.path.join(download_path, f"{safe_name}_clipped.tif")

states = gpd.read_file(SHAPEFILE_PATH)

state_boundary = states[
    states["State_Name"].str.strip().str.lower()
    == state_name.strip().lower()
]

temp_geojson = os.path.join(download_path, "temp_boundary.geojson")

state_boundary.to_file(temp_geojson, driver="GeoJSON")

print("Clipping and compressing directly from VRT...")

gdal.Warp(
    output_raster,
    vrt_file,
    cutlineDSName=temp_geojson,
    cropToCutline=True,
    dstNodata=0,
    creationOptions=[
        "COMPRESS=DEFLATE",
        "PREDICTOR=2",
        "BIGTIFF=YES",
        "TILED=YES"
    ]
)

os.remove(temp_geojson)

print("Done!")
print(output_raster)

Clipping and compressing directly from VRT...
Done!
C:\Users\Subham V Sharma\Desktop\Python geospatial\satellite-downloader\satellite-imagery-downloader\sikkim_Landsat-8\sikkim_clipped.tif


In [155]:
import os

folder = download_path

keep_file = f"{safe_name}_clipped.tif"

for file in os.listdir(folder):

    if file != keep_file:

        path = os.path.join(folder, file)

        if os.path.isfile(path):

            os.remove(path)
            print("Deleted:", file)

print("Finished")

Deleted: sikkim.vrt
Deleted: tile_1.tif
Deleted: tile_1.vrt
Deleted: tile_1_chunk_1.tif
Deleted: tile_2.tif
Deleted: tile_2.vrt
Deleted: tile_2_chunk_1.tif
Deleted: tile_3.tif
Deleted: tile_3.vrt
Deleted: tile_3_chunk_1.tif
Deleted: tile_4.tif
Deleted: tile_4.vrt
Deleted: tile_4_chunk_1.tif
Deleted: tile_5.tif
Deleted: tile_5.vrt
Deleted: tile_5_chunk_1.tif
Deleted: tile_6.tif
Deleted: tile_6.vrt
Deleted: tile_6_chunk_1.tif
Finished


In [156]:
from osgeo import gdal
import os

safe_name = state_name.replace(" ", "_")

output_raster = os.path.join(
    download_path,
    f"{safe_name}_clipped.tif"
)

print("Building overviews...")

image = gdal.Open(output_raster, gdal.GA_Update)

image.BuildOverviews(
    "AVERAGE",
    [2, 4, 8, 16, 32, 64]
)

image = None  # close the file

print("Done! Overviews built.")

Building overviews...
Done! Overviews built.
